In [87]:
%load_ext autoreload
%autoreload 2

In [89]:
import os
import sys
from pathlib import Path
import glob

project_root = Path.cwd().parent
sys.path.append(str(project_root))

import pandas as pd
import numpy as np

# Experimental conditions

In [154]:
silver_path = Path.cwd().parent / 'data/silver_layer/'
silver_confocal_path = Path.cwd().parent / 'data/silver_layer/confocal_runs'
save_path = Path.cwd().parent / 'data/silver_layer/confocal_runs_processed'

save_path.mkdir(parents=True, exist_ok=True)

In [155]:
df_experimental = pd.read_pickle(silver_path / 'experimental_results.pkl')

In [156]:
def calculate_confocal_stats(
    confocal_path: Path,
    threshold: float,
) -> pd.DataFrame:
    """
    Calculate summary statistics for all confocal runs considering
    only t1 values below a given threshold.
    """

    stats_list = []

    for csv_path in sorted(confocal_path.glob("confocal_run_*.csv")):
        df_runs = pd.read_csv(csv_path)

        t1_valid = df_runs["t1"].dropna()
        t1_filtered = t1_valid[t1_valid < threshold]

        stats = {
            "threshold": threshold,
            "file": csv_path.name,
            "run_id": int(csv_path.stem.replace("confocal_run_", "")),
            "n_total": len(df_runs),
            "n_t1_valid": len(t1_valid),
            "n_t1_missing": df_runs["t1"].isna().sum(),
            "n_t1_below_threshold": len(t1_filtered),
            "fraction_below_threshold": (
                len(t1_filtered) / len(t1_valid)
                if len(t1_valid) > 0
                else None
            ),
            "min": t1_filtered.min(),
            "max": t1_filtered.max(),
            "mean": t1_filtered.mean(),
            "median": t1_filtered.median(),
            "q10": t1_filtered.quantile(0.10),
            "q25": t1_filtered.quantile(0.25),
            "q75": t1_filtered.quantile(0.75),
            "q90": t1_filtered.quantile(0.90),

        }

        stats_list.append(stats)

    stats_df = pd.DataFrame(stats_list)
    stats_df.sort_values(by="run_id", inplace=True)

    return stats_df

In [157]:
def calculate_t2_stats(
    confocal_path: Path,
) -> pd.DataFrame:

    decided_threshold = 0.985
    decided_second_threshold = 0.373

    n_water = 1.33
    n_bk7 = 1.51

    correction_factor = n_water / n_bk7


    stats_list = []

    for csv_path in sorted(confocal_path.glob("confocal_run_*.csv")):

        df = pd.read_csv(csv_path)

        t2 = df["t2"].dropna()
        d2 = (
            (df["t1"] - decided_threshold)
            .clip(lower=0)
            * correction_factor
        )

        missing_mask = df["t2"].isna()
        adjusted_values = d2.loc[missing_mask]

        stats = {
            "file": csv_path.name,
            "run_id": int(csv_path.stem.replace("confocal_run_", "")),

            "n_total": len(df),

            "missing_rate_t1":
                (
                    df["t1"].isna().sum()
                    / df.shape[0]
                    * 100
                ),

            "missing_rate_t2":
                (
                    df["t2"].isna().sum()
                    / df.shape[0]
                    * 100
                ),

            "adjusted_thickness_mean": adjusted_values.mean(),
            "adjusted_thickness_median": adjusted_values.median(),
            "adjusted_thickness_min": adjusted_values.min(),
            "adjusted_thickness_max": adjusted_values.max(),

            "min_t2": t2.min(),
            "mean_t2": t2.mean(),
            "median_t2": t2.median(),
            "max_t2": t2.max(),

            "q05_t2": t2.quantile(0.05),
            "q10_t2": t2.quantile(0.10),
            "q25_t2": t2.quantile(0.25),
            "q75_t2": t2.quantile(0.75),
            "q90_t2": t2.quantile(0.90),
        }

        stats_list.append(stats)

    stats_df = pd.DataFrame(stats_list)
    stats_df.sort_values("run_id", inplace=True)

    return stats_df

In [158]:
thresholds = [0.965, 0.97, 0.98, 0.985, 0.99, 0.995, 1.0, 1.05, 1.1, 1.5]

metrics = [
    "min",
    "max",
    "mean",
    "median",
    "q10",
    "q25",
    "q75",
    "q90",
    "fraction_below_threshold",
]

rows = {}

for threshold in thresholds:

    stats_df = calculate_confocal_stats(
        silver_confocal_path,
        threshold
    )

    for metric in metrics:
        rows.setdefault(f"{metric}_mean", {})[threshold] = stats_df[metric].mean()
        rows.setdefault(f"{metric}_std", {})[threshold] = stats_df[metric].std()

threshold_comparison = pd.DataFrame(rows).T
threshold_comparison = threshold_comparison.round(3)

In [159]:
threshold_comparison[[0.97, 0.98, 0.985, 0.99, 1.05]]

,0.970,0.980,0.985,0.990,1.050
min_mean,0.962,0.963,0.963,0.963,0.963
min_std,0.005,0.005,0.005,0.005,0.005
max_mean,0.970,0.980,0.985,0.989,1.040
max_std,0.000,0.000,0.001,0.001,0.019
mean_mean,0.968,0.975,0.976,0.977,0.983
mean_std,0.001,0.002,0.003,0.003,0.013
median_mean,0.969,0.975,0.976,0.976,0.982
median_std,0.001,0.002,0.003,0.003,0.013
q10_mean,0.967,0.971,0.972,0.972,0.973
q10_std,0.001,0.002,0.003,0.003,0.007


In [160]:
stats_t2 = calculate_t2_stats(silver_confocal_path)

In [161]:
df2 = pd.merge(stats_t2, df_experimental[['run_id', 'reduced_pattern']], on='run_id', how='inner')

In [162]:
adj_cols = ['adjusted_thickness_mean', 'adjusted_thickness_median','adjusted_thickness_max']

In [163]:
df2.groupby('reduced_pattern')[adj_cols].agg(['min', 'max', 'mean', 'median']).round(3)

adjusted_thickness_mean                       \
                                    min    max   mean median   
reduced_pattern                                                
annular                           0.021  0.069  0.044  0.044   
bubbly                            0.000  0.000  0.000  0.000   
churn                             0.015  0.078  0.037  0.032   
dispersed                         0.000  0.029  0.010  0.005   
plug                              0.000  0.176  0.034  0.002   
slug                              0.000  0.083  0.024  0.009   
stratified                        0.001  0.251  0.105  0.051   
wavy                              0.000  0.255  0.057  0.056   

                adjusted_thickness_median                       \
                                      min    max   mean median   
reduced_pattern                                                  
annular                             0.016  0.056  0.039  0.039   
bubbly                              0.000  0.000  0.000  0.000   
churn                               0.000  0.047  0.008  0.000   
dispersed                           0.000  0.000  0.000  0.000   
plug                                0.000  0.200  0.031  0.000   
slug                                0.000  0.000  0.000  0.000   
stratified                          0.000  0.265  0.101  0.040   
wavy                                0.000  0.300  0.040  0.039   

                adjusted_thickness_max                       
                                   min    max   mean median  
reduced_pattern                                              
annular                          0.066  0.397  0.180  0.135  
bubbly                           0.002  0.221  0.037  0.007  
churn                            0.263  0.664  0.483  0.449  
dispersed                        0.000  0.517  0.270  0.322  
plug                             0.000  0.813  0.263  0.190  
slug                             0.062  0.729  0.451  0.429  
stratified                       0.074  0.613  0.376  0.418  
wavy                             0.082  0.697  0.433  0.423

In [164]:
df2.sort_values(by='q05_t2').head(5)[['missing_rate_t2', 'min_t2', 'q05_t2', 'q10_t2', 'reduced_pattern']].round(3)

,missing_rate_t2,min_t2,q05_t2,q10_t2,reduced_pattern
6,87.60,0.265,0.304,0.316,wavy
10,87.69,0.273,0.306,0.321,wavy
11,91.27,0.252,0.307,0.321,wavy
1,96.41,0.287,0.314,0.326,annular
8,81.92,0.276,0.321,0.345,dispersed


In [165]:
df2.query('missing_rate_t2<50').sort_values(by='q05_t2').head(5)[['missing_rate_t2', 'min_t2', 'q05_t2', 'q10_t2','reduced_pattern']].round(3)

,missing_rate_t2,min_t2,q05_t2,q10_t2,reduced_pattern
33,37.21,0.343,0.373,0.381,plug
35,30.47,0.344,0.381,0.398,plug
32,23.32,0.342,0.395,0.412,plug
34,43.14,0.328,0.400,0.415,plug
46,43.34,0.351,0.418,0.438,wavy


In [168]:
decided_threshold = 0.985
decided_second_threshold = 0.373

n_water = 1.33
n_bk7 = 1.51

correction_factor = n_water / n_bk7

In [ ]:
# Process all files
for file in silver_confocal_path.glob("confocal_run_*.csv"):

    df = pd.read_csv(file)

    # Apply correction
    excess = (df["t1"] - decided_threshold).clip(lower=0)
    df.loc[(df["t2"].isna()) & (excess * correction_factor > decided_second_threshold), "t2"] = 0
    df["t1"] = df["t1"].clip(upper=decided_threshold)
    df["t2"] = df["t2"] + excess * correction_factor

    # Save with same name
    output_file = save_path / file.name
    df.to_csv(output_file, index=False)

    print(f"Processed: {file.name}")

Processed: confocal_run_1.csv
Processed: confocal_run_10.csv
Processed: confocal_run_100.csv
Processed: confocal_run_101.csv
Processed: confocal_run_102.csv
Processed: confocal_run_103.csv
Processed: confocal_run_104.csv
Processed: confocal_run_105.csv
Processed: confocal_run_106.csv
Processed: confocal_run_107.csv
Processed: confocal_run_108.csv
Processed: confocal_run_109.csv
Processed: confocal_run_11.csv
Processed: confocal_run_110.csv
Processed: confocal_run_111.csv
Processed: confocal_run_112.csv
Processed: confocal_run_113.csv
Processed: confocal_run_114.csv
Processed: confocal_run_115.csv
Processed: confocal_run_116.csv
Processed: confocal_run_117.csv
Processed: confocal_run_118.csv
Processed: confocal_run_119.csv
Processed: confocal_run_12.csv
Processed: confocal_run_120.csv
Processed: confocal_run_121.csv
Processed: confocal_run_122.csv
Processed: confocal_run_123.csv
Processed: confocal_run_124.csv
Processed: confocal_run_125.csv
Processed: confocal_run_126.csv
Processed: co

In [173]:
thresholds = [
    0,
    0.050,
    0.100,
    0.150,
    0.200,
    0.250,
    0.300,
    0.350,
    0.373,
    0.400,
    0.450,
]

rows = []

for second_threshold in thresholds:

    total_rows = 0
    total_missing_t2 = 0
    total_created_t2 = 0

    for file in silver_confocal_path.glob("confocal_run_*.csv"):

        df = pd.read_csv(file)

        excess = (df["t1"] - decided_threshold).clip(lower=0)
        transferred_thickness = excess * correction_factor

        total_rows += len(df)

        total_missing_t2 += df["t2"].isna().sum()

        mask = (
            df["t2"].isna()
            & (transferred_thickness > second_threshold)
        )

        total_created_t2 += mask.sum()

    rows.append({
        "second_threshold": second_threshold,
        "total_rows": total_rows,
        "missing_t2": total_missing_t2,
        "created_t2": total_created_t2,
        "created_pct_of_missing":
            100 * total_created_t2 / total_missing_t2,
        "remaining_missing":
            total_missing_t2 - total_created_t2,
        "remaining_pct_of_missing":
            100 * (total_missing_t2 - total_created_t2)
            / total_missing_t2,
    })

summary_df = pd.DataFrame(rows).round(2)

In [174]:
summary_df

,second_threshold,total_rows,missing_t2,created_t2,created_pct_of_missing,remaining_missing,remaining_pct_of_missing
0,0.00,1640000,1111742,269010,24.20,842732,75.80
1,0.05,1640000,1111742,161958,14.57,949784,85.43
2,0.10,1640000,1111742,98428,8.85,1013314,91.15
3,0.15,1640000,1111742,62390,5.61,1049352,94.39
4,0.20,1640000,1111742,37777,3.40,1073965,96.60
5,0.25,1640000,1111742,22536,2.03,1089206,97.97
6,0.30,1640000,1111742,12775,1.15,1098967,98.85
7,0.35,1640000,1111742,6085,0.55,1105657,99.45
8,0.37,1640000,1111742,3565,0.32,1108177,99.68
9,0.40,1640000,1111742,1532,0.14,1110210,99.86
